# 03 — Model Tuning, Calibration & Threshold Selection

**Project:** Hospital Readmission Prediction  
**Input:** Best baseline from `02_modeling_baseline.ipynb`

| # | Section |
|---|---|
| 1 | Environment setup |
| 2 | Load features + splits |
| 3 | Hyperparameter tuning |
| 4 | Evaluate tuned models |
| 5 | Probability calibration |
| 6 | Threshold optimisation |
| 7 | Final evaluation on test set |
| 8 | Save tuned model |

---
## 1 — Environment setup

In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import load_config, get_logger, set_seed, save_model, load_model

config = load_config(ROOT / 'config' / 'config.yaml')
set_seed(config['random_seed'])
config['_base_dir'] = str(ROOT)
config['_figures_subfolder'] = 'tuning'

logger = get_logger('03_model_tuning')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print(f'ROOT: {ROOT}')
print('Environment ready.')

ROOT: C:\Users\speak\HA_Project\Hospital-Readmission-Project-
Environment ready.


---
## 2 — Load features + splits

In [2]:
from src.modeling import load_features, make_splits

X, y, feature_names = load_features(config, base_dir=ROOT)
X_train, X_val, X_test, y_train, y_val, y_test = make_splits(X, y, config)

print(f'Train: {len(y_train):,} | Val: {len(y_val):,} | Test: {len(y_test):,}')
print(f'Positive rate — Train: {y_train.mean():.3f} | Val: {y_val.mean():.3f} | Test: {y_test.mean():.3f}')

2026-03-18 11:33:25 | INFO     | src.modeling | Loaded feature dataset: 25000 rows × 48 columns


2026-03-18 11:33:25 | INFO     | src.modeling | Feature matrix: 25000 rows × 47 features | pos=11754 (47.0%) neg=13246 (53.0%)


2026-03-18 11:33:25 | INFO     | src.modeling | Split: train=15000 | val=5000 | test=5000


Train: 15,000 | Val: 5,000 | Test: 5,000
Positive rate — Train: 0.470 | Val: 0.470 | Test: 0.470


---
## 3 — Hyperparameter tuning

Tune all three baselines with `RandomizedSearchCV` (20 iterations, 5-fold stratified CV, scoring = ROC-AUC).

In [3]:
from src.modeling import build_baselines, build_param_distributions, tune_model

models = build_baselines(config)
tuned_results = {}

for name, model in models.items():
    print(f'\nTuning {name}...')
    param_dist = build_param_distributions(name, config)
    if not param_dist:
        print(f'  No param space defined for {name} — skipping tuning.')
        model.fit(X_train, y_train)
        tuned_results[name] = {'estimator': model, 'best_params': {}, 'cv_score': None}
        continue
    result = tune_model(model, param_dist, X_train, y_train, config)
    tuned_results[name] = result
    print(f'  Best CV ROC-AUC: {result["cv_score"]:.4f}')
    print(f'  Best params: {result["best_params"]}')

2026-03-18 11:33:25 | INFO     | src.modeling | Baseline estimators: ['LogisticRegression', 'RandomForest', 'GradientBoosting']


2026-03-18 11:33:25 | INFO     | src.modeling | RandomizedSearchCV: model=Pipeline  n_iter=20  cv=5  scoring=roc_auc



Tuning LogisticRegression...


2026-03-18 11:33:57 | INFO     | src.modeling | Best roc_auc: 0.6452 | params: {'classifier__solver': 'saga', 'classifier__penalty': 'l1', 'classifier__C': 0.1}


2026-03-18 11:33:57 | INFO     | src.modeling | RandomizedSearchCV: model=RandomForestClassifier  n_iter=20  cv=5  scoring=roc_auc


...

2026-03-18 11:34:30 | INFO     | src.modeling | Best roc_auc: 0.6564 | params: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 10, 'max_depth': 15}


2026-03-18 11:34:30 | INFO     | src.modeling | RandomizedSearchCV: model=HistGradientBoostingClassifier  n_iter=20  cv=5  scoring=roc_auc


  Best CV ROC-AUC: 0.6564
  Best params: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 10, 'max_depth': 15}

Tuning GradientBoosting...


2026-03-18 11:34:46 | INFO     | src.modeling | Best roc_auc: 0.6558 | params: {'min_samples_leaf': 30, 'max_iter': 300, 'max_depth': 3, 'learning_rate': 0.2}


  Best CV ROC-AUC: 0.6558
  Best params: {'min_samples_leaf': 30, 'max_iter': 300, 'max_depth': 3, 'learning_rate': 0.2}


---
## 4 — Evaluate tuned models on validation set

In [4]:
from src.modeling import evaluate, compare_models, select_best_model, plot_roc_curves, plot_pr_curves, plot_confusion_matrix

tuned_estimators = {name: res['estimator'] for name, res in tuned_results.items()}

eval_tuned = {}
for name, model in tuned_estimators.items():
    metrics = evaluate(model, X_val, y_val, threshold=0.5)
    eval_tuned[name] = metrics
    print(f'{name}: ROC-AUC={metrics["roc_auc"]:.4f}  Recall={metrics["recall"]:.4f}  '
          f'Precision={metrics["precision"]:.4f}  F1={metrics["f1"]:.4f}')

summary_tuned = compare_models(eval_tuned)
display(summary_tuned)

best_tuned_name = select_best_model(summary_tuned, metric='roc_auc')
best_tuned_model = tuned_estimators[best_tuned_name]
print(f'\nBest tuned model: {best_tuned_name}')

LogisticRegression: ROC-AUC=0.6510  Recall=0.5206  Precision=0.6135  F1=0.5633
RandomForest: ROC-AUC=0.6557  Recall=0.5751  Precision=0.5956  F1=0.5852
GradientBoosting: ROC-AUC=0.6578  Recall=0.5619  Precision=0.5959  F1=0.5784


,roc_auc,pr_auc,accuracy,precision,recall,f1,specificity,brier
model,,,,,,,,
GradientBoosting,0.6578,0.6354,0.6148,0.5959,0.5619,0.5784,0.6618,0.2303
RandomForest,0.6557,0.6354,0.6166,0.5956,0.5751,0.5852,0.6535,0.2305
LogisticRegression,0.6510,0.6279,0.6204,0.6135,0.5206,0.5633,0.7089,0.2320


2026-03-18 11:34:46 | INFO     | src.modeling | Best model by roc_auc: GradientBoosting (0.6578)



Best tuned model: GradientBoosting


In [5]:
# Confusion matrices for all tuned models
for name, model in tuned_estimators.items():
    safe = name.lower().replace(' ', '_')
    plot_confusion_matrix(
        model, X_val, y_val,
        model_name=name, config=config,
        threshold=0.5,
        filename=f'confusion_matrix_tuned_{safe}.png'
    )

plot_roc_curves(tuned_estimators, X_val, y_val, config,
                title='ROC Curves — Tuned Models',
                filename='roc_curve_tuned_models.png')
plot_pr_curves(tuned_estimators, X_val, y_val, config,
               title='Precision-Recall — Tuned Models',
               filename='pr_curve_tuned_models.png')

2026-03-18 11:34:46 | INFO     | src.modeling | Saved confusion matrix → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\tuning\confusion_matrix_tuned_logisticregression.png


2026-03-18 11:34:46 | INFO     | src.modeling | Saved confusion matrix → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\tuning\confusion_matrix_tuned_randomforest.png


2026-03-18 11:34:46 | INFO     | src.modeling | Saved confusion matrix → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\tuning\confusion_matrix_tuned_gradientboosting.png


2026-03-18 11:34:46 | INFO     | src.modeling | Saved ROC curve → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\tuning\roc_curve_tuned_models.png


2026-03-18 11:34:47 | INFO     | src.modeling | Saved PR curve → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\tuning\pr_curve_tuned_models.png


---
## 5 — Probability calibration

Calibrators are fitted on a 70% sub-split of the validation set; Brier scores are evaluated on the held-out 30% (avoiding self-referential comparison). The best method is then refit on the full validation set.

In [6]:
from src.modeling import calibrate_model

calibrated_model = calibrate_model(
    best_tuned_model, X_val, y_val, config, save_plot=True
)
print(f'Calibrated model type: {type(calibrated_model).__name__}')
print(f'Calibration method applied: {calibrated_model.method}')

2026-03-18 11:34:47 | INFO     | src.modeling | Calibration (sigmoid): Brier=0.2279 (held-out eval)


2026-03-18 11:34:47 | INFO     | src.modeling | Calibration (isotonic): Brier=0.2287 (held-out eval)


2026-03-18 11:34:47 | INFO     | src.modeling | Best calibration method: sigmoid


2026-03-18 11:34:47 | INFO     | src.modeling | Saved calibration comparison → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\tuning\calibration_curve.png


Calibrated model type: _PrefitCalibratedModel
Calibration method applied: sigmoid


In [7]:
# Compare calibrated vs uncalibrated on validation set
from sklearn.metrics import brier_score_loss

prob_uncal = best_tuned_model.predict_proba(X_val)[:, 1]
prob_cal   = calibrated_model.predict_proba(X_val)[:, 1]

print(f'Brier score — Uncalibrated: {brier_score_loss(y_val, prob_uncal):.4f}')
print(f'Brier score — Calibrated  : {brier_score_loss(y_val, prob_cal):.4f}')
print(f'\nROC-AUC — Uncalibrated: {evaluate(best_tuned_model, X_val, y_val)["roc_auc"]:.4f}')
print(f'ROC-AUC — Calibrated  : {evaluate(calibrated_model, X_val, y_val)["roc_auc"]:.4f}')

Brier score — Uncalibrated: 0.2303
Brier score — Calibrated  : 0.2296

ROC-AUC — Uncalibrated: 0.6578
ROC-AUC — Calibrated  : 0.6578


---
## 6 — Threshold optimisation

Strategy: find the **highest-precision threshold** where recall ≥ 0.80. This avoids the low-threshold collapse seen with highly imbalanced datasets. With ≈45% positive rate, meaningful thresholds are expected.

In [8]:
from src.modeling import threshold_sweep, plot_threshold_analysis

sweep_df, optimal_threshold = threshold_sweep(calibrated_model, X_val, y_val, config)
calibrated_model.threshold = float(optimal_threshold)

print(f'Optimal threshold: {optimal_threshold:.3f}')
print(f'At optimal threshold:')
opt_row = sweep_df[sweep_df['threshold'] == optimal_threshold].iloc[0]
print(f'  Precision: {opt_row["precision"]:.4f}')
print(f'  Recall   : {opt_row["recall"]:.4f}')
print(f'  F1       : {opt_row["f1"]:.4f}')
print(f'  Specificity: {opt_row["specificity"]:.4f}')

2026-03-18 11:34:47 | INFO     | src.modeling | Threshold sweep: optimal threshold by recall = 0.350


Optimal threshold: 0.350
At optimal threshold:
  Precision: 0.5119
  Recall   : 0.8766
  F1       : 0.6464
  Specificity: 0.2582


In [9]:
plot_threshold_analysis(sweep_df, optimal_threshold, config,
                         filename='threshold_analysis.png')

# Confusion matrix at optimal threshold
plot_confusion_matrix(
    calibrated_model, X_val, y_val,
    model_name=f'{best_tuned_name} (calibrated)',
    config=config,
    threshold=optimal_threshold,
    filename='confusion_matrix_calibrated_optimal.png'
)

2026-03-18 11:34:47 | INFO     | src.modeling | Saved threshold analysis → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\tuning\threshold_analysis.png


2026-03-18 11:34:48 | INFO     | src.modeling | Saved confusion matrix → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\tuning\confusion_matrix_calibrated_optimal.png


---
## 7 — Final evaluation on held-out test set

In [10]:
test_metrics = evaluate(calibrated_model, X_test, y_test, threshold=optimal_threshold)

print('Test set metrics at optimal threshold:')
for k, v in test_metrics.items():
    print(f'  {k:<15}: {v:.4f}')

Test set metrics at optimal threshold:
  roc_auc        : 0.6534
  pr_auc         : 0.6233
  accuracy       : 0.5484
  precision      : 0.5115
  recall         : 0.8792
  f1             : 0.6467
  specificity    : 0.2548
  brier          : 0.2309


In [11]:
# Save tuned metrics summary
metrics_tuned_path = ROOT / config['paths']['metrics_tuned']

# Include best tuned model metrics (val + test)
val_metrics = evaluate(calibrated_model, X_val, y_val, threshold=calibrated_model.threshold)
test_metrics = evaluate(calibrated_model, X_test, y_test, threshold=calibrated_model.threshold)

metrics_summary = pd.DataFrame({
    'val_set':  val_metrics,
    'test_set': test_metrics,
}).T
metrics_summary.index.name = 'split'
display(metrics_summary.round(4))

metrics_summary.to_csv(metrics_tuned_path)
print(f'Saved tuned metrics to {metrics_tuned_path}')

,roc_auc,pr_auc,accuracy,precision,recall,f1,specificity,brier
split,,,,,,,,
val_set,0.6578,0.6354,0.5490,0.5119,0.8766,0.6464,0.2582,0.2296
test_set,0.6534,0.6233,0.5484,0.5115,0.8792,0.6467,0.2548,0.2309


Saved tuned metrics to C:\Users\speak\HA_Project\Hospital-Readmission-Project-\data\processed\metrics_tuned_summary.csv


---
## 8 — Save tuned model

In [12]:
model_path = ROOT / config['paths']['model_dir'] / 'best_tuned_model.pkl'
save_model({
    'model':     calibrated_model,
    'threshold': calibrated_model.threshold,
    'model_name': best_tuned_name,
    'val_metrics': val_metrics,
    'test_metrics': test_metrics,
    'feature_columns': feature_names,
}, model_path)

print(f'Saved calibrated model + metadata to {model_path}')
print(f'\nSummary:')
print(f'  Model       : {best_tuned_name} (calibrated: {calibrated_model.method})')
print(f'  Threshold   : {calibrated_model.threshold:.3f}')
print(f'  Val ROC-AUC : {val_metrics["roc_auc"]:.4f}')
print(f'  Test ROC-AUC: {test_metrics["roc_auc"]:.4f}')
print(f'  Test Recall : {test_metrics["recall"]:.4f}')
print(f'  Test Precision: {test_metrics["precision"]:.4f}')
print('\nNext: run 04_model_interpretation_and_error_analysis.ipynb')

Saved calibrated model + metadata to C:\Users\speak\HA_Project\Hospital-Readmission-Project-\models\best_tuned_model.pkl

Summary:
  Model       : GradientBoosting (calibrated: sigmoid)
  Threshold   : 0.350
  Val ROC-AUC : 0.6578
  Test ROC-AUC: 0.6534
  Test Recall : 0.8792
  Test Precision: 0.5115

Next: run 04_model_interpretation_and_error_analysis.ipynb
